In [1]:
import pandas as pd

In [2]:
comments = pd.read_csv('/kaggle/input/datasets/mcchainiy/comments-labels/comments.csv')

Обучаем токенайзер

In [3]:
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False) as f:
    for comment in comments['comment']:
        f.write(comment + '\n')
    temp_path = f.name

In [4]:
from tokenizers import BertWordPieceTokenizer

tokenizer = BertWordPieceTokenizer(lowercase=True)

tokenizer.train(
    files=[temp_path],
    vocab_size=30000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
)

In [5]:
tokenizer.save_model("tokenizer")

Exception: No such file or directory (os error 2)

In [ ]:
tokenizer.save("tokenizer/tokenizer.json")

Загружаем токенизатор

In [6]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("./tokenizer")

Обучаем берт

In [ ]:
from transformers import BertConfig

config = BertConfig(
    vocab_size=30000,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=512,
    max_position_embeddings=512
)

In [ ]:
from transformers import BertForMaskedLM

model = BertForMaskedLM(config=config)

Dataset for mlm

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "text",
    data_files={"train": temp_path}
)

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Маскируем датасет

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

Обучаем модель

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./custom_bert",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=5e-4,
    save_steps=1000,
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

trainer.train()

In [ ]:
trainer.save_model("./custom_bert")
tokenizer.save_pretrained("./custom_bert")

Готовимся к классификации

In [8]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "./bert",
    num_labels=4
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
comments = pd.read_csv("/Users/danil/Desktop/toxic-comments/data/comments.csv")

In [15]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
comments['label'] = le.fit_transform(comments['label'])

In [21]:
comments.to_csv('../../data/comments_labels_num.csv', index=False)

In [28]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files="/Users/danil/Desktop/toxic-comments/data/comments_labels_num.csv")
dataset = dataset.class_encode_column("label")

Stringifying the column:   0%|          | 0/248281 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/248281 [00:00<?, ? examples/s]

In [ ]:
dataset = dataset["train"].train_test_split(test_size=0.2, stratify_by_column='label')

In [31]:
def tokenize_classification(examples):
    return tokenizer(
        examples["comment"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize_classification, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/198624 [00:00<?, ? examples/s]

Map:   0%|          | 0/49657 [00:00<?, ? examples/s]

In [33]:
train_dataset = dataset["train"]
val_dataset = dataset["test"]

Обучаем классификацию

In [34]:
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    preds = np.argmax(logits, axis=1)
    
    return {
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds)
    }

In [37]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_output",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    logging_steps=100
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

/Users/danil/miniforge3/envs/datascience/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
100,0.904800
200,0.619400
300,0.615900
400,0.625600
500,0.634200
600,0.586100
700,0.606300
800,0.589700
900,0.583100
1000,0.554400


/Users/danil/miniforge3/envs/datascience/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/danil/miniforge3/envs/datascience/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=37242, training_loss=0.25205809781569416, metrics={'train_runtime': 3310.4215, 'train_samples_per_second': 179.999, 'train_steps_per_second': 11.25, 'total_flos': 995686895812608.0, 'train_loss': 0.25205809781569416, 'epoch': 3.0})

In [40]:
trainer.save_model("bert-clf")
tokenizer.save_pretrained("bert-clf")

('bert-clf/tokenizer_config.json',
 'bert-clf/special_tokens_map.json',
 'bert-clf/vocab.txt',
 'bert-clf/added_tokens.json',
 'bert-clf/tokenizer.json')

Загружаем обученную модель

In [43]:
from transformers import BertForSequenceClassification, BertTokenizerFast
import torch

model_path = "bert-clf"

tokenizer = BertTokenizerFast.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

In [98]:
label_map = {
    0: "INSULT",
    1: "NORMAL",
    2: "THREAT",
    3: "OBSCENITY"
}

In [ ]:
# def bert_predict_comment(text):
#     inputs = tokenizer(
#         text,
#         return_tensors="pt",
#         truncation=True,
#         padding=True,
#         max_length=128
#     )
    
#     with torch.no_grad():
#         outputs = model(**inputs)
    
#     logits = outputs.logits
#     pred = torch.argmax(logits, dim=1).item()
    
#     return label_map[pred]

# def bert_predict(texts):
#     predictions = []
#     for i in texts:
#         predictions.append(bert_predict_comment(i))
#     return predictions

In [101]:
import torch
import tqdm 

def bert_predict_series(texts, batch_size=32):
    model.eval()
    
    all_preds = []
    
    texts = texts.tolist()
    
    for i in tqdm.tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        
        all_preds.extend(preds)
    
    return [label_map[p] for p in all_preds]

Используем модель

In [92]:
from sklearn import model_selection
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import seaborn as sns

In [93]:
df = pd.read_csv('../../data/comments.csv')
X = df['comment']
y = df['label']

# X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.3, 
#                                                                     random_state=42, stratify=y)

In [94]:
def print_multiclass_metrics(y_test, y_pred, model_name='LR'):
    print("f1 macro score:", f1_score(y_test, y_pred, average='macro'))
    print("f1 weighted score:", f1_score(y_test, y_pred, average='weighted'))
    print(f"{model_name} accurcay:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred, target_names=['INSULT', 'NORMAL', 'THREAT', 'OBSCENITY']))


    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=df['label'].unique(), yticklabels=df['label'].unique())

In [ ]:
y_pred = bert_predict_series(X)

 28%|██▊       | 2203/7759 [02:21<07:00, 13.20it/s]

In [ ]:
print_multiclass_metrics(y, y_pred)